In [ ]:
# Initial imports
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

# Get the parent directory 
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
# Import custom modules
import Scripts.sql_server_connect as sql
import Scripts.python_cleaning as clean

QUERIES_DIR = project_root / "Queries" / "SQL Business queries"
ASSETS_DIR = project_root / "Assets"
ASSETS_DIR.mkdir(exist_ok=True)

# Database settings
# Set OLIST_SERVER and OLIST_DATABASE as environment variables to override these defaults.
SERVER = os.getenv("OLIST_SERVER", r"localhost\SQLEXPRESS")
DATABASE = os.getenv("OLIST_DATABASE", "OlistDB")

# Column names returned by the SQL files (change here if you rename them in SQL)
CAT_NAME_COL = "category_name"
GMV_COL = "gross_merchandise_value"   
AOV_COL = "average_order_value"
ORDERS_COL = "order_count"

TARGET = "Health & Beauty"

# Colours
OLIST_COLORS = {
    "primary_blue": "#3951db",
    "dark_navy": "#131b31",
    "coral_orange": "#d45d37",
    "gold": "#dfa344",
    "sage_green": "#88a86a",
    "mauve": "#d2abc2",
}

# Plot style: set once, applies to every chart below
plt.rcParams.update({
    "figure.figsize": (15, 5),
    "figure.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 18,
    "axes.titleweight": "bold",
    "axes.titlepad": 15,
    "axes.titlecolor": OLIST_COLORS["gold"],
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    "xtick.labelsize": 11,
    "xtick.color": OLIST_COLORS["gold"],
    "ytick.color": OLIST_COLORS["gold"],
    "legend.frameon": False,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.transparent": True,
})


In [ ]:
# Connecting to MSSQL Server database
engine = sql.mssql_engine(SERVER, DATABASE)

5.3 Product Analysis

In [ ]:
#Load products dataset
df_product = sql.load_sql_to_df(str(QUERIES_DIR/"SQL Products.sql"), engine)
# Adding revenue share and cumulative revenue share columns for the Pareto chart.
df_product["revenue_share"] = df_product["revenue"] / df_product["revenue"].sum() * 100
df_product["revenue_share_cumsum"] = df_product["revenue_share"].cumsum()
df_product["label"] = "Product " + (df_product.index + 1).astype(str)
#Describe price and size of products
descrs = [(df_product[col].describe()) for col in df_product[["avg_price","volumetric_weight_kg"]]]
descrs = pd.DataFrame(descrs).T

In [ ]:
print(descrs)

Our price and weight data is rightly skewed, as it is common for money and size values.
To flag high values in a right-skewed data, we will first log-transform them and then calculate IQR.(https://pmc.ncbi.nlm.nih.gov/articles/PMC9036143/)

In [ ]:
##Price distribution
fig, ax = plt.subplots()

ax.hist(df_product["avg_price"], bins = 45, linewidth=0.5, edgecolor="white",density=True)
ax.set_xlabel("Value")
ax.set_xticks(range(0,2000,200))
ax.set_ylabel("Count")
ax.set_title("Basic Histogram")
plt.show()

In [ ]:
# Recalculate quantile bands from the products loaded for the selected category and period.
def quantile_band(values, q=3):
    values = pd.to_numeric(values, errors="coerce")
    valid = values.dropna()
    bands = pd.Series(pd.NA, index=values.index, dtype="string")
    if valid.empty:
        return bands
    if valid.nunique() == 1:
        bands.loc[valid.index] = "Single value"
        return bands
    intervals = pd.qcut(valid, q=q, duplicates="drop")
    bands.loc[valid.index] = intervals.astype("string").to_numpy()
    return bands


df_product["price_band"] = quantile_band(df_product["avg_price"])
df_product["size_band"] = quantile_band(df_product["volumetric_weight_kg"])

# Flag high prices using a dynamic IQR fence in log space; keep original prices and revenue unchanged.
log_price = np.log1p(df_product["avg_price"].where(df_product["avg_price"] >= 0))
price_q1, price_q3 = log_price.quantile([0.25, 0.75])
price_upper_fence = np.expm1(price_q3 + 1.5 * (price_q3 - price_q1))
df_product["price_high_outlier"] = df_product["avg_price"] > price_upper_fence

In [ ]:
##Price distribution
fig, ax = plt.subplots()

ax.hist(log_price, bins = 45, linewidth=0.5, edgecolor="white",density=True)
ax.set_xlabel("Value")
#ax.set_xticks(range(0,2000,200))
ax.set_ylabel("Count")
ax.set_title("Log Transformed Histogram")
plt.show()

In [ ]:
#Aggrregate Table about outliers.
df_product_outliers = df_product.loc[df_product["price_high_outlier"]].copy()

units_sold = df_product_outliers["units_sold"].sum()
summary = pd.DataFrame([{
    "Flagged products": df_product_outliers["product_id"].nunique(),
    "Units sold": units_sold,
    "Min price (R$)": df_product_outliers["min_price"].min(),
    "Weighted avg price (R$)": (
        (df_product_outliers["avg_price"] * df_product_outliers["units_sold"]).sum() / units_sold
        if units_sold else np.nan
    ),
    "Max price (R$)": df_product_outliers["max_price"].max(),
    "Avg actual weight (kg)": df_product_outliers["actual_weight_kg"].mean(),
    "Avg volumetric weight (kg)": df_product_outliers["volumetric_weight_kg"].mean(),
    "Max volumetric weight (kg)": df_product_outliers["volumetric_weight_kg"].max(),
    "Revenue (R$)": df_product_outliers["revenue"].sum(),
    "Revenue share (%)": df_product_outliers["revenue_share"].sum()
}])


styled = (
    summary.style
    .format({
        "Flagged products": "{:,.0f}",
        "Units sold": "{:,.0f}",
        "Min price (R$)": "R$ {:,.2f}",
        "Weighted avg price (R$)": "R$ {:,.2f}",
        "Max price (R$)": "R$ {:,.2f}",
        "Avg actual weight (kg)": "{:,.3f}",
        "Avg volumetric weight (kg)": "{:,.3f}",
        "Max volumetric weight (kg)": "{:,.3f}",
        "Revenue (R$)": "R$ {:,.2f}",
        "Revenue share (%)": "{:.2f}%",
    }, na_rep="-")
    .hide(axis="index")
)

display(styled)

Our upper fence for the outlier price starts from the 849.90 reais. And there is 36 products above flagged as outliers. What is interesting, that they account for only 76 sold units, but make 10.55% of revenue share.

Since we don't have subcategories or any additional information about the product except the category name. I am dividing data into bands by price and by size. Each value will have 3 bands, representing Low/Mid/High value divided by quartiles in the underlying distribution.
Which combination of price/size generated most of the revenue?

In [ ]:
segment_summary = (
    df_product.groupby(["price_band", "size_band"], observed=True)
    .agg(
        revenue=("revenue", "sum"),
        products=("product_id", "nunique"),
        units_sold=("units_sold", "sum"),
    )
    .reset_index()
)

# Sort bands by their actual underlying values, not by interval-label text.
price_order = (
    df_product.groupby("price_band", observed=True)["avg_price"]
    .mean()
    .sort_values()
    .index
)
size_order = (
    df_product.groupby("size_band", observed=True)["volumetric_weight_kg"]
    .mean()
    .sort_values()
    .index
)

revenue_grid = (
    segment_summary.pivot(
        index="price_band", columns="size_band", values="revenue"
    )
    .reindex(index=price_order, columns=size_order)
)

products_grid = (
    segment_summary.pivot(
        index="price_band", columns="size_band", values="products"
    )
    .reindex(index=price_order, columns=size_order)
)

units_grid = (
    segment_summary.pivot(
        index="price_band", columns="size_band", values="units_sold"
    )
    .reindex(index=price_order, columns=size_order)
)

In [ ]:
revenue_grid

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
fig, ax = plt.subplots()
revenue_grid = revenue_grid.fillna(0)
heat_grid = revenue_grid.to_numpy()
vmin = float(np.nanmin(heat_grid))
vmax = float(np.nanmax(heat_grid))
if vmin == vmax:
    vmax = vmin + 1

olist_red_green = LinearSegmentedColormap.from_list(
    "olist_red_green",
    [
        OLIST_COLORS["coral_orange"],
        "#F7F5F0",
        OLIST_COLORS["sage_green"],
    ],
)
norm = Normalize(vmin=vmin, vmax=vmax)

im = ax.imshow(heat_grid, cmap=olist_red_green, norm=norm)

# Show all ticks and label them with the respective list entries
ax.set_xticks(np.arange(len(revenue_grid.columns)), labels=revenue_grid.columns,
              rotation=45, rotation_mode="xtick")
ax.set_yticks(np.arange(len(revenue_grid.index)), labels=revenue_grid.index)


ax.set_title("Revenue by Product Price and Size", pad=14, weight="bold")
ax.set_xlabel("Size band (volumetric weight)")
ax.set_ylabel("Price band (average item price)")

# Subtle cell borders make the segments easier to compare.
ax.set_xticks(np.arange(-0.5, len(revenue_grid.columns), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(revenue_grid.index), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)

for i in range(revenue_grid.shape[0]):
    for j in range(revenue_grid.shape[1]):
        revenue = heat_grid[i, j]
        r, g, b, _ = olist_red_green(norm(revenue))
        luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
        text_color = OLIST_COLORS["dark_navy"] if luminance > 0.55 else "white"
        ax.text(
            j,
            i,
            f"R$ {revenue:,.0f}\n",
            ha="center",
            va="center",
            color=text_color,
            fontsize=9,
            fontweight="bold",
        )

cbar = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.03)
cbar.set_label("Revenue (R$)")

plt.show()
